# Module: Adversarial Search

Adversarial search chooses moves in a game against an opponent who is trying to make you lose.
Instead of a path to a goal, it looks for the best move, assuming the opponent also plays their best
reply.

| # | Algorithm | Explores |
|---|---|---|
| 1 | Minimax | the whole game tree, depth-first |
| 2 | Alpha-Beta Pruning | the same tree, skipping branches that cannot change the decision |

Games use the same dict style as the graphs before. `tree[n]` lists the positions reachable from n
in one move, and a second dict `utility[n]` scores each leaf from MAX's point of view. MAX moves at
the root and the players alternate. Each function returns `(value, best_move, nodes_visited)`,
where a node counts as visited when the search reaches it.

## Games as search

A two-player, turn-taking, zero-sum game with perfect information (tic-tac-toe, chess, Go) is a
search problem with two agents:

- **Initial state** s<sub>0</sub>, and **Player(s)**: whose turn it is in s.
- **Actions(s)** and **Result(s, a)**: the legal moves and the position each one leads to.
- **Terminal-Test(s)**: whether the game is over.
- **Utility(s)**: the final score of a terminal state for MAX. The game is *zero-sum*: whatever MAX
  gains, MIN loses, so MIN wants the utility as low as possible.

One move by one player is a **ply**. The **minimax value** of a state is the utility MAX ends up with
if both players play optimally from there:

$$\text{minimax}(s) = \begin{cases} \text{Utility}(s) & \text{if } s \text{ is terminal} \\ \max_{a} \ \text{minimax}(\text{Result}(s, a)) & \text{if MAX moves in } s \\ \min_{a} \ \text{minimax}(\text{Result}(s, a)) & \text{if MIN moves in } s \end{cases}$$

**Real games are too big to search to the end.** Chess has about 35 legal moves per position and a
game lasts about 80 plies, so the full tree has around 35<sup>80</sup> ≈ 10<sup>123</sup> nodes.
Programs stop at a fixed depth and score the positions there with an **evaluation function**, for
example a material count in chess. The leaf values in this notebook play that role.

<h2><b>1. Minimax</b></h2>

![](data\minimax.gif)

GIF source: generated for this module with matplotlib (the case-study tree below)

Minimax searches the game tree depth-first. A leaf returns its utility, a MAX node returns the
largest value among its children and a MIN node the smallest. The values back up level by level to
the root, where MAX takes the move with the highest value.

It keeps only the recursion stack: the path from the root to the current node. Unlike DFS in the
uninformed notebook, it has no goal to stop at. It must visit every node, because any leaf it skips
could change a value on the way up.

**Step-by-step mechanism of Minimax:**

- At the root (MAX), compute the value of every child and return the child with the largest value
  as the best move.
- To compute the value of a node:
  - If it is a leaf, return its utility.
  - If MAX moves there, return the maximum of its children's values.
  - If MIN moves there, return the minimum of its children's values.
- There is no failure case: every finite game tree has a minimax value.

**Properties of Minimax**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes, if the tree is finite | Yes, against an optimal opponent | O(b<sup>m</sup>) | O(bm) |

- **Complete?** It searches the whole finite tree, so it always returns a value and a move.
- **Optimal?** It maximises the worst case. Against an opponent who makes mistakes a riskier move could score more, but minimax never scores less than its value.
- **Time:** it visits every node; with branching factor b and depth m that is b<sup>m</sup> leaves.
- **Space:** depth-first, so it stores one path of length m and the children of each node on it.

**Case study.** Rina is playing a board game against her brother Bayu. She looks two plies ahead:
her move (**B**, **C** or **D**), then Bayu's reply. Each leaf is Rina's lead in points after both
moves; Bayu wants it as small as possible.

In [1]:
rina_tree = {
    'A': ['B', 'C', 'D'],       # Rina (MAX) to move
    'B': ['B1', 'B2', 'B3'],    # Bayu (MIN) replies
    'C': ['C1', 'C2', 'C3'],
    'D': ['D1', 'D2', 'D3'],
}

rina_utility = {
    'B1': 3, 'B2': 12, 'B3': 8,
    'C1': 2, 'C2': 4,  'C3': 6,
    'D1': 14, 'D2': 5, 'D3': 2,
}

In [2]:
import math

def minimax(tree, utility, root, verbose=False):
    """Minimax on an explicit game tree, MAX to move at the root.
    Returns (value, best_move, nodes_visited)."""
    nodes_visited = 0

    def value(node, maximizing, depth):
        nonlocal nodes_visited
        nodes_visited += 1
        if node not in tree:                      # leaf: terminal or depth-limit position
            return utility[node]
        child_values = [value(child, not maximizing, depth + 1) for child in tree[node]]
        best = max(child_values) if maximizing else min(child_values)
        if verbose:
            print(f"{'    ' * depth}{'MAX' if maximizing else 'MIN'} {node}: {child_values} -> {best}")
        return best

    nodes_visited += 1                            # the root
    best_move, best_value = None, -math.inf
    for child in tree[root]:
        child_value = value(child, False, 1)
        if child_value > best_value:
            best_move, best_value = child, child_value
    if verbose:
        print(f"MAX {root}: best move {best_move} -> {best_value}")
    return best_value, best_move, nodes_visited

In [3]:
value, move, visited = minimax(rina_tree, rina_utility, 'A', verbose=True)
print()
print("Best move     :", move)
print("Minimax value :", value)
print("Nodes visited :", visited)

    MIN B: [3, 12, 8] -> 3
    MIN C: [2, 4, 6] -> 2
    MIN D: [14, 5, 2] -> 2
MAX A: best move B -> 3

Best move     : B
Minimax value : 3
Nodes visited : 13


Move D leads to the best leaf (14), but Bayu would answer with D3 and leave Rina only 2. Minimax
assumes the worst reply to every move and picks B, which guarantees a lead of at least 3. It visits
all 13 nodes of the tree.

<h2><b>2. Alpha-Beta Pruning</b></h2>

![](data\alpha_beta.gif)

GIF source: generated for this module with matplotlib (the case-study tree below)

Alpha-beta pruning returns the same value and move as minimax but skips subtrees that cannot change
the decision. Each call carries two bounds from the path above it:

- **α**: the best value MAX can already guarantee somewhere on the path (a lower bound).
- **β**: the best value MIN can already guarantee somewhere on the path (an upper bound).

When α ≥ β at a node, one of the players above already has a better option elsewhere and will never
let the game reach this node, so its remaining children are skipped. In Rina's tree, B is worth 3,
so MAX can already get 3. The first leaf under C is 2, so C is worth at most 2 and MAX will never
choose it; C2 and C3 need not be looked at.

The data is the same recursion stack as minimax, plus α and β in each call. How much it prunes
depends on the order in which the children are searched.

**Step-by-step mechanism of Alpha-Beta Pruning:**

- Start at the root with α = −∞ and β = +∞.
- At a leaf, return its utility.
- At a MAX node, for each child in turn:
  - Compute the child's value with the current α and β.
  - Update best = max(best, value) and α = max(α, best).
  - If α ≥ β, skip the remaining children (**β cut-off**) and return best.
- At a MIN node, the same with best = min(best, value) and β = min(β, best); if α ≥ β, skip the
  rest (**α cut-off**).
- At the root, the child that last raised α is the best move. As with minimax, there is no failure
  case.

**Properties of Alpha-Beta Pruning**

| Complete? | Optimal? | Time | Space |
|---|---|---|---|
| Yes, if the tree is finite | Yes, same value and move as minimax | O(b<sup>m/2</sup>) with perfect ordering, O(b<sup>m</sup>) worst case | O(bm) |

- **Complete?** It only skips subtrees; everything else is searched as in minimax.
- **Optimal?** A pruned subtree cannot change the value at the root, so the result is exactly the minimax value.
- **Time:** if the best move is always tried first, it examines about b<sup>m/2</sup> leaves, so it can search twice as deep as minimax in the same time; with random ordering it is about b<sup>3m/4</sup>; with the worst ordering it prunes nothing.
- **Space:** the same depth-first stack as minimax.

**Case study.** Pak Hendra's chess app looks three plies ahead: his move (**B**, **C** or **D**), his
opponent's reply, and his follow-up. Each leaf is the app's evaluation of the resulting position, in
pawns, from Pak Hendra's side.

In [4]:
chess_tree = {
    'A': ['B', 'C', 'D'],                                     # Pak Hendra (MAX)
    'B': ['E', 'F'], 'C': ['G', 'H'], 'D': ['I', 'J'],        # opponent (MIN)
    'E': ['E1', 'E2'], 'F': ['F1', 'F2'], 'G': ['G1', 'G2'],  # Pak Hendra (MAX)
    'H': ['H1', 'H2'], 'I': ['I1', 'I2'], 'J': ['J1', 'J2'],
}

chess_utility = {
    'E1': 4, 'E2': 6, 'F1': 7, 'F2': 9,
    'G1': 1, 'G2': 2, 'H1': 0, 'H2': -1,
    'I1': 9, 'I2': 7, 'J1': 8, 'J2': 2,
}

In [5]:
import math

def alpha_beta(tree, utility, root, verbose=False):
    """Minimax with alpha-beta pruning, MAX to move at the root.
    Returns (value, best_move, nodes_visited)."""
    nodes_visited = 0

    def value(node, alpha, beta, maximizing, depth):
        nonlocal nodes_visited
        nodes_visited += 1
        if node not in tree:                      # leaf: terminal or depth-limit position
            return utility[node]
        children = tree[node]
        best = -math.inf if maximizing else math.inf
        for k, child in enumerate(children):
            child_value = value(child, alpha, beta, not maximizing, depth + 1)
            if maximizing:
                best = max(best, child_value)
                alpha = max(alpha, best)
            else:
                best = min(best, child_value)
                beta = min(beta, best)
            if alpha >= beta:                     # the player above will never allow this node
                if verbose and k + 1 < len(children):
                    print(f"{'    ' * depth}{'MAX' if maximizing else 'MIN'} {node}: "
                          f"alpha = {alpha}, beta = {beta} -> prune {children[k + 1:]}")
                break
        if verbose:
            print(f"{'    ' * depth}{'MAX' if maximizing else 'MIN'} {node} = {best}")
        return best

    nodes_visited += 1                            # the root
    alpha, best_move = -math.inf, None
    for child in tree[root]:
        child_value = value(child, alpha, math.inf, False, 1)
        if child_value > alpha:
            alpha, best_move = child_value, child
    if verbose:
        print(f"MAX {root}: best move {best_move} -> {alpha}")
    return alpha, best_move, nodes_visited

In [6]:
value, move, visited = alpha_beta(chess_tree, chess_utility, 'A', verbose=True)
print()
print("Best move     :", move)
print("Minimax value :", value)
print("Nodes visited :", visited)
print("Minimax alone :", minimax(chess_tree, chess_utility, 'A')[2], "nodes")

        MAX E = 6
        MAX F: alpha = 7, beta = 6 -> prune ['F2']
        MAX F = 7
    MIN B = 6
        MAX G = 2
    MIN C: alpha = 6, beta = 2 -> prune ['H']
    MIN C = 2
        MAX I = 9
        MAX J = 8
    MIN D = 8
MAX A: best move D -> 8

Best move     : D
Minimax value : 8
Nodes visited : 18
Minimax alone : 22 nodes


B comes back as 6, so α = 6 at the root. Inside F the first leaf is already 7: MIN at B has 6 and
would never let the game into F, so F2 is skipped. Under C, G is worth only 2, so C is worth at most
2 < 6 and the whole subtree H is skipped. D still has to be searched in full, and it wins with 8.
Alpha-beta returns the same move and value as minimax while visiting 18 of the 22 nodes.

## Comparison

Both algorithms on one larger random game tree: branching factor 3, depth 6 (729 leaves, 1,093
nodes). Alpha-beta runs three times on the same leaves, changing only the order of the children:

- **as generated:** the order the tree was built in.
- **best first:** at every node, the best child for the player to move comes first.
- **worst first:** the best child comes last.

Sorting by the true minimax values is only possible here because we can compute them in advance;
real programs estimate a good order, for example with the evaluation function or the result of a
shallower search.

In [7]:
import random

def make_game_tree(branching, depth, seed=None):
    """Uniform random game tree with integer leaf utilities in [-20, 20]. Returns (tree, utility)."""
    rng = random.Random(seed)
    tree, utility = {}, {}

    def grow(node, level):
        if level == depth:
            utility[node] = rng.randint(-20, 20)
            return
        tree[node] = [node + str(i) for i in range(branching)]
        for child in tree[node]:
            grow(child, level + 1)

    grow('R', 0)
    return tree, utility

def reorder(tree, utility, root, best_first=True):
    """Copy of tree whose children are sorted by their minimax value for the player to move."""
    values = {}

    def value(node, maximizing):
        if node not in tree:
            values[node] = utility[node]
        else:
            child_values = [value(child, not maximizing) for child in tree[node]]
            values[node] = max(child_values) if maximizing else min(child_values)
        return values[node]

    value(root, True)
    ordered, stack = {}, [(root, True)]
    while stack:
        node, maximizing = stack.pop()
        if node in tree:
            ordered[node] = sorted(tree[node], key=values.get, reverse=(maximizing == best_first))
            stack.extend((child, not maximizing) for child in tree[node])
    return ordered

In [8]:
def print_table(rows):
    header = f"{'algorithm':<28} | {'best move':>9} | {'value':>5} | {'nodes visited':>13}"
    print(header)
    print("-" * len(header))
    for name, move, value, visited in rows:
        print(f"{name:<28} | {move:>9} | {value:>5} | {visited:>13}")

game_tree, game_utility = make_game_tree(branching=3, depth=6, seed=0)

value, move, visited = minimax(game_tree, game_utility, 'R')
rows = [('Minimax', move, value, visited)]
for label, best_first in [('as generated', None), ('best first', True), ('worst first', False)]:
    tree = game_tree if best_first is None else reorder(game_tree, game_utility, 'R', best_first)
    value, move, visited = alpha_beta(tree, game_utility, 'R')
    rows.append((f'Alpha-Beta ({label})', move, value, visited))
print_table(rows)

algorithm                    | best move | value | nodes visited
----------------------------------------------------------------
Minimax                      |        R2 |    -8 |          1093
Alpha-Beta (as generated)    |        R2 |    -8 |           293
Alpha-Beta (best first)      |        R2 |    -8 |           125
Alpha-Beta (worst first)     |        R2 |    -8 |           949


All four runs pick R2 with value −8: pruning never changes the answer. Minimax visits all 1,093
nodes. Alpha-beta visits 293 with the generated order, and only 125 when the best child always comes
first. That run looks at 53 of the 729 leaves, the theoretical minimum
b<sup>⌈m/2⌉</sup> + b<sup>⌊m/2⌋</sup> − 1 = 27 + 27 − 1. With the worst child first it still visits
949 nodes, nearly as many as minimax. The algorithm and the leaves are the same in every run; only
the order changes. This is why game programs spend effort on move ordering, for example trying
captures first or the best move from the previous, shallower search.